In [1]:
import sys

from absl import logging
from ferminet.utils import system
from ferminet import base_config
from ferminet import train
from ferminet.configs import atom

# Optional, for also printing training progress to STDOUT.
# If running a script, you can also just use the --alsologtostderr flag.
logging.get_absl_handler().python_handler.stream = sys.stdout
logging.set_verbosity(logging.INFO)


# Define H2 molecule
cfg = base_config.default()
cfg.system.electrons = (1,1)  # (alpha electrons, beta electrons)
cfg.system.molecule = [system.Atom('H', (0, 0, -1)), system.Atom('H', (0, 0, 1))]

# Set training parameters
cfg.batch_size = 4096
cfg.pretrain.iterations = 0
cfg.mcmc.burn_in = 0
cfg.optim.optimizer = 'minsr'


In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
evaluate_loss, mcmc_step, sharded_key, data, params, mcmc_width, logabs_network = train.train(cfg, wandb_monitoring=False)

INFO:absl:Starting QMC with 1 XLA devices per host across 1 hosts.


cfg.optim.optimizer = 'minsr'


INFO:absl:No checkpoint found. Training new model.
INFO:absl:Burning in MCMC chain for 0 steps
INFO:absl:Completed burn-in MCMC steps
INFO:absl:Initial energy: -1.8168 E_h


In [ ]:
train.train(cfg, wandb_monitoring=False)

### Constructing S and QGT through streaming gradients

In [6]:
import jax
import jax.numpy as jnp
from jax import tree_util
import gc
from tqdm.notebook import tqdm

In [7]:
from ferminet import constants

In [10]:
# squeezing pmap dimension to work on 1 gpu
data_positions = jnp.squeeze(data.positions, axis=0)
data_spins = jnp.squeeze(data.spins, axis=0)
data_atoms = jnp.squeeze(data.atoms, axis=0)
data_charges = jnp.squeeze(data.charges, axis=0)
params_ = jax.tree.map(lambda x: jnp.squeeze(x, axis=0), params)

Define a jacobian vector product here based on params

And then a second one for the second vector X(XT)x

In [8]:
batch_network = jax.vmap(
      logabs_network, in_axes=(None, 0, 0, 0, 0), out_axes=0
  ) # Multi sample network output

grad_params = jax.grad(logabs_network, argnums=0)  # Single sample grad wrt model output

batch_network_grad = jax.vmap(
      grad_params, in_axes=(None, 0, 0, 0, 0), out_axes=0
  )  # Multi sample grad wrt model output (memory problems very quickly)

In [11]:
# Flatten once to capture treedef and shapes
leaves, treedef = jax.tree_util.tree_flatten(params_)
shapes = [leaf.shape for leaf in leaves]
sizes = [leaf.size for leaf in leaves]

def flat_to_pytree(flat_vec):
    idx = 0
    new_leaves = []
    for shape, size in zip(shapes, sizes):
        new_leaves.append(flat_vec[idx:idx + size].reshape(shape))
        idx += size
    return jax.tree_util.tree_unflatten(treedef, new_leaves)


In [12]:
def params_to_array(params_pytree, batched=False):
    leaves, _ = jax.tree_util.tree_flatten(params_pytree)
    if batched:
        batch_size = leaves[0].shape[0]
        flat_leaves = [jnp.reshape(leaf, (batch_size, -1)) for leaf in leaves]
        return jnp.concatenate(flat_leaves, axis=1)  # Shape: (batch_size, n_params)
    else:
        return jnp.concatenate([jnp.ravel(leaf) for leaf in leaves], axis=0)

### First jacfwd product 

In [13]:
def f(p):
    return batch_network(p, data_positions, data_spins, data_atoms, data_charges)

In [ ]:
y, f_jvp = jax.linearize(f, x)

In [16]:
n_params = sum(jnp.size(p) for p in jax.tree_util.tree_leaves(params))

In [34]:
jvp_func = lambda x: jax.linearize(f, params_)[1](flat_to_pytree(x))

In [40]:
vjp_func_minsr = lambda x: params_to_array(jax.vjp(f, params_)[1](x))

In [45]:
jvp_func_minsr = lambda x: params_to_array(jax.linearize(f, params_)[1](flat_to_pytree(x)))

## Second jacrev product

In [29]:
vjp_func = lambda v: params_to_array(jax.vjp(f, params_)[1](v))

In [59]:
def minsr_matmul(v, centre_gradients=True, damping=1e-4, batch_size=4096):
    log_psi_jac_v = vjp_func_minsr(v) / batch_size
    update_vector = jvp_func_minsr(log_psi_jac_v)

    #if centre_gradients:
    #    update_vector -= jnp.mean(update_vector)
    
    #update_vector += damping * v
    return update_vector

In [47]:
key = jax.random.PRNGKey(0)
n = 4096
x = jax.random.normal(key, shape=(n,))
t = minsr_matmul(x)

In [48]:
len(t)

667104

In [51]:
evaluate_loss = constants.pmap(evaluate_loss)

In [52]:
loss, aux_data = evaluate_loss(params, sharded_key, data)

In [53]:
energies = aux_data.local_energy - loss

In [56]:
energies = jnp.squeeze(energies, axis=0)

In [57]:
energies.shape

(4096,)

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax

# Direct solve using dense construction (for testing / small problems)
def direct_solve(matvec, b):
    # Build dense matrix A by applying matvec to each basis vector
    n = b.shape[0]
    eye = jnp.eye(n)
    A = jnp.stack([matvec(eye[i]) for i in range(n)], axis=1)
    return jnp.linalg.solve(A, b)

def direct_transpose_solve(matvec, b):
    # Same trick but for A^T
    n = b.shape[0]
    eye = jnp.eye(n)
    A = jnp.stack([matvec(eye[i]) for i in range(n)], axis=1)
    return jnp.linalg.solve(A.T, b)

# Wrapper for linear solve
def minsr_solve(b):
    return lax.custom_linear_solve(
        matvec=minsr_matmul,
        solve=direct_solve,
        transpose_solve=direct_transpose_solve
    )(b)


In [ ]:
minsr_solve(energies)

In [60]:
grads = vjp_func_minsr(jax.lax.custom_linear_solve(
          minsr_matmul, energies, solve=jax.numpy.linalg.solve)[0]
        )

TypeError: jnp.linalg.solve requires ndarray or scalar arguments, got <class 'function'> at position 0.

In [28]:
s = jvp_func_minsr(t[0])

In [36]:
len(s)

4096

In [37]:
l = params_to_array(vjp_func_minsr(s))

In [38]:
len(l)

667104

In [23]:
s = params_to_array(t)

1

In [22]:
t

({'envelope': [{'pi': Array([[ 1.2032539e+02, -1.9133144e+01, -5.5239856e+02, -3.5334648e+01,
             4.8574771e+02, -1.0800943e+02,  4.0113318e+02, -4.2247198e+02,
             1.5371459e+03,  7.8859436e+02,  9.0131421e+02,  2.7316000e+02,
            -4.3387032e+01, -1.9349857e+02, -7.2468738e+02,  3.9292670e+02,
            -7.9465955e+02,  3.2221561e+02, -8.9763550e+01,  8.5549362e+01,
            -1.0831888e+02, -6.3681641e+02,  7.5857196e+02,  3.8709698e+01,
             2.0364254e+01, -2.1363740e+03,  4.8258789e+01,  7.2531319e+01,
            -1.3722282e+02, -1.3129782e+01,  8.4199486e+01, -3.8365778e+02],
           [ 1.6944414e+01,  1.4786656e+01, -1.9131065e+02, -2.2857887e+01,
             1.7335522e+02, -3.3359093e+01,  1.3770564e+02, -1.2430662e+02,
             5.4924640e+02,  2.2622186e+02,  3.1143481e+02,  9.1015381e+01,
            -6.7918687e+00, -8.2080849e+01, -2.8273157e+02,  1.3698764e+02,
            -2.8313684e+02,  1.3496432e+02, -2.0337671e+01,  3.056148

In [ ]:
t.shape

In [32]:
def fisher_matmul(v, centre_gradients=True, damping=1e-4, batch_size=4096):
    log_psi_jac_v = jvp_func(v)
    update_vector = vjp_func(log_psi_jac_v / batch_size)

    if centre_gradients:
        update_vector -= jnp.mean(update_vector)
    
    update_vector += damping * v
    return update_vector

In [33]:
key = jax.random.PRNGKey(0)
n = n_params
x = jax.random.normal(key, shape=(n,))

t = fisher_matmul(x)

### Getting b loss grads

In [35]:
loss_and_grad = jax.value_and_grad(evaluate_loss, argnums=0, has_aux=True)

In [36]:
lg = constants.pmap(loss_and_grad)

In [37]:
(loss, aux_data), grad = lg(params, sharded_key, data)

In [38]:
flat_grads = params_to_array(grad)

In [42]:
flat_params, unravel_fun = jax.flatten_util.ravel_pytree(grad)

[autoreload of ferminet.train failed: Traceback (most recent call last):
  File "/mnt/c/Users/Parv/Doc/RA/Projects/QVMC/fermi/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/mnt/c/Users/Parv/Doc/RA/Projects/QVMC/fermi/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/parv/anaconda3/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 991, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1129, in get_code
  File "<frozen importlib._bootstrap_external>", line 1059, in source_to_code
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/mnt/c/Users/Parv/Doc/RA/Projects/ferminet/ferminet/tra

In [43]:
flat_params.shape

(667104,)

In [49]:
outie = jax.scipy.sparse.linalg.cg(fisher_matmul, flat_grads,  maxiter=100)

In [51]:
outie[0]

Array([ 4.6242433e-04,  4.7416593e-06, -1.0764901e-04, ...,
        2.0735723e-04,  1.7751299e-04, -4.4789471e-05], dtype=float32)

In [31]:
from tqdm.notebook import tqdm

In [ ]:
jvp_func = lambda x: jax.jvp(batch_network, (params_,), (x,))[1]

In [7]:
def params_to_array(params_pytree, batched=False):
    leaves, _ = jax.tree_util.tree_flatten(params_pytree)
    if batched:
        batch_size = leaves[0].shape[0]
        flat_leaves = [jnp.reshape(leaf, (batch_size, -1)) for leaf in leaves]
        return jnp.concatenate(flat_leaves, axis=1)  # Shape: (batch_size, n_params)
    else:
        return jnp.concatenate([jnp.ravel(leaf) for leaf in leaves], axis=0)


In [31]:
def clear_memory():
    jax.clear_caches()
    gc.collect()
    jax.device_put(jax.numpy.zeros(1)).block_until_ready()

In [36]:
grad_params = val_grad(params_, data_positions[0], data_spins[0], data_atoms[0], data_charges[0])

In [43]:
# limits of batched based gradients
n_samples = 64

param_grads = batch_network_grad(
    params_, data_positions[:n_samples], data_spins[:n_samples], data_atoms[:n_samples], data_charges[:n_samples])

In [8]:
def get_batch_network_gradients(pos_start, pos_end):
    return batch_network_grad(
        params_,
        data_positions[pos_start:pos_end], data_spins[pos_start: pos_end], data_atoms[pos_start: pos_end], data_charges[pos_start: pos_end])

In [57]:
def A(x, n_params):
    """ Takes in the vector x (N_samples) and uses it as a premultiplier for the gradients """
    output_vec = jnp.zeros(n_params)

    minibatch = 64
    for i in range(0, n_samples, minibatch):
        pos_start = i * minibatch
        pos_end = i * minibatch + minibatch
        output_vec = output_vec.at[:].add(
            get_batch_network_gradients(pos_start, pos_end) @ x[pos_start:pos_end])

    return output_vec

In [ ]:
n_params = params_to_array(params_).shape[0]
key = jax.random.PRNGKey(0)
n = 4096
x = jax.random.normal(key, shape=(n,))
output_vec = jnp.zeros(n_params)

minibatch = 4
for i in tqdm(range(0, n, minibatch)):
    pos_start = i * minibatch
    pos_end = i * minibatch + minibatch
    output_vec = output_vec.at[:].add(
        params_to_array(get_batch_network_gradients(pos_start, pos_end), batched=True).T @ x[pos_start:pos_end])